In [2]:
import os
import sys
from pathlib import Path
from uuid import uuid4
from shutil import copyfile
from tempfile import TemporaryDirectory
from unittest.mock import patch

from dotenv import load_dotenv
from langchain_core.messages import AIMessage
from langchain_core.tracers.context import collect_runs
from langchain_google_genai import ChatGoogleGenerativeAI
from langsmith import Client, traceable
from langgraph.types import Command
from openevals.llm import create_llm_as_judge
from openevals.prompts import CORRECTNESS_PROMPT
from agentevals.trajectory.match import create_trajectory_match_evaluator

from langgraph_finance_agent.graph import main_graph

load_dotenv()
print("LANGSMITH_API_KEY 설정 여부:", bool(os.getenv("LANGSMITH_API_KEY")))

MODEL_NAME = "gemini-3.6-flash"
model = ChatGoogleGenerativeAI(model=MODEL_NAME)


LANGSMITH_API_KEY 설정 여부: True


In [5]:
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from langgraph_finance_agent.nodes import classify_category

CATEGORIES = ["account", "bill_payment", "card", "product", "unsupported"]

# expected: 문자열 하나 또는 허용 목록(리스트). 리스트면 그중 아무거나 나와도 정답.
router_set = [
    # ── account ↔ product ──
    {"q": "정기예금 계좌 만들고 싶어", "expected": "product", "tag": "acct_vs_prod"},
    {"q": "적금 통장 하나 개설해줘", "expected": "product", "tag": "acct_vs_prod"},
    {"q": "예금 계좌 새로 열고 싶은데", "expected": ["product", "account"], "tag": "acct_vs_prod"},
    {"q": "대출받고 싶은데 돈 받을 통장은 월급통장으로 해줘", "expected": "product", "tag": "acct_vs_prod"},
    {"q": "월급통장 하나 만들어줘", "expected": "account", "tag": "acct_vs_prod"},  # 대조군
    {"q": "입출금 계좌 개설할래요", "expected": "account", "tag": "acct_vs_prod"},  # 대조군

    # ── account ↔ bill_payment: 단서가 명확한 것 ──
    {"q": "가스비 자동이체 걸어줘", "expected": "bill_payment", "tag": "bill_clear"},
    {"q": "자동이체 목록 보여줘", "expected": "bill_payment", "tag": "bill_clear"},
    {"q": "전기세 자동이체 해지해줘", "expected": "bill_payment", "tag": "bill_clear"},
    {"q": "생활비 계좌에서 저축 계좌로 5만원 보내줘", "expected": "account", "tag": "bill_clear"},  # 대조군

    # ── account ↔ bill_payment: "계좌"만 있고 모호한 것 (둘 다 허용, 흐름은 셀 2에서 확인) ──
    {"q": "전기세 계좌 만들고 싶어", "expected": ["account", "bill_payment"], "tag": "bill_ambiguous"},
    {"q": "관리비 내는 계좌 만들어줘", "expected": ["account", "bill_payment"], "tag": "bill_ambiguous"},
    {"q": "수도요금 납부용 계좌 하나 필요해", "expected": ["account", "bill_payment"], "tag": "bill_ambiguous"},

    # ── account ↔ card ──
    {"q": "체크카드 연결 계좌 바꿔줘", "expected": "card", "tag": "acct_vs_card"},
    {"q": "데일리카드로 5만원 결제했어", "expected": "card", "tag": "acct_vs_card"},
    {"q": "카드 만들 때 월급통장에 연결해줘", "expected": "card", "tag": "acct_vs_card"},
    {"q": "월급통장 잔액 알려줘", "expected": "account", "tag": "acct_vs_card"},  # 대조군

    # ── product / card 기타 ──
    {"q": "신용대출 300만원 12개월로 신청할게", "expected": "product", "tag": "prod_misc"},
    {"q": "내 적금 해지하면 얼마 받아?", "expected": "product", "tag": "prod_misc"},
    {"q": "신용카드 결제일이 언제야?", "expected": "card", "tag": "prod_misc"},

    # ── 해지 동사: 도메인은 명사가 결정 ──
    {"q": "여행통장 해지해줘", "expected": "account", "tag": "verb_confuse"},
    {"q": "데일리카드 해지해줘", "expected": "card", "tag": "verb_confuse"},
    {"q": "적금 해지해줘", "expected": "product", "tag": "verb_confuse"},

    # ── unsupported ──
    # 입금은 account로 가도 account 안의 unsupported 액션이 막아주므로 둘 다 허용 (최종 응답은 셀 2에서 확인)
    {"q": "내 통장에 100만원 입금해줘", "expected": ["unsupported", "account"], "tag": "unsupported"},
    {"q": "달러로 환전해줘", "expected": "unsupported", "tag": "unsupported"},
    {"q": "종합소득세 납부하고 싶어", "expected": "unsupported", "tag": "unsupported"},
    {"q": "오늘 날씨 어때?", "expected": "unsupported", "tag": "unsupported"},
    {"q": "삼성전자 주식 사줘", "expected": "unsupported", "tag": "unsupported"},

    # ── 표현 변형 ──
    {"q": "전기요금 자동납부 설정", "expected": "bill_payment", "tag": "variation"},
    {"q": "한전 요금 자동이체 신청할게요", "expected": "bill_payment", "tag": "variation"},
    {"q": "정기예금 가입", "expected": "product", "tag": "variation"},
    {"q": "예금 가입하고싶은데 계좌만들어야하나요", "expected": "product", "tag": "variation"},
    {"q": "체크 카드 발급좀", "expected": "card", "tag": "variation"},
    {"q": "내 balance 확인해줘", "expected": "account", "tag": "variation"},
]


def route(q: str) -> str:
    try:
        return classify_category({"request": q})["category"]
    except Exception as e:
        return f"ERROR:{type(e).__name__}"


def evaluate_router(dataset, max_workers=8):
    questions = [case["q"] for case in dataset]
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        routed_list = list(ex.map(route, questions))

    rows = []
    for case, routed in zip(dataset, routed_list):
        allowed = case["expected"] if isinstance(case["expected"], list) else [case["expected"]]
        rows.append({**case, "allowed": allowed, "routed": routed, "correct": routed in allowed})
    return rows


rows = evaluate_router(router_set)

# 1) 전체 / 태그별 정확도
by_tag = defaultdict(list)
for r in rows:
    by_tag[r["tag"]].append(r["correct"])

print(f"전체 정확도: {sum(r['correct'] for r in rows) / len(rows):.1%} ({sum(r['correct'] for r in rows)}/{len(rows)})")
for tag, results in sorted(by_tag.items()):
    print(f"  {tag:15s} {sum(results) / len(results):6.1%}  (n={len(results)})")

# 2) 혼동 행렬 (행=정답, 허용 목록이면 첫 번째 / 열=라우팅 결과)
cm = Counter((r["allowed"][0], r["routed"]) for r in rows)
print("\n행=정답 / 열=라우팅 결과")
print(f"{'':14s}" + "".join(f"{c[:8]:>10s}" for c in CATEGORIES))
for exp in CATEGORIES:
    print(f"{exp:14s}" + "".join(f"{cm[(exp, c)]:>10d}" for c in CATEGORIES))

# 3) 오답 + 모호 케이스의 실제 라우팅 결과
print("\n[오답]")
for r in rows:
    if not r["correct"]:
        print(f"- ({r['tag']}) {r['q']}\n    허용={r['allowed']}  라우팅 결과={r['routed']}")

print("\n[모호 케이스가 실제로 어디로 갔는지]")
for r in rows:
    if len(r["allowed"]) > 1:
        print(f"- {r['q']} → {r['routed']}")

전체 정확도: 100.0% (34/34)
  acct_vs_card    100.0%  (n=4)
  acct_vs_prod    100.0%  (n=6)
  bill_ambiguous  100.0%  (n=3)
  bill_clear      100.0%  (n=4)
  prod_misc       100.0%  (n=3)
  unsupported     100.0%  (n=5)
  variation       100.0%  (n=6)
  verb_confuse    100.0%  (n=3)

행=정답 / 열=라우팅 결과
                 account  bill_pay      card   product  unsuppor
account                9         0         0         0         0
bill_payment           0         5         0         0         0
card                   0         0         6         0         0
product                1         0         0         8         0
unsupported            1         0         0         0         4

[오답]

[모호 케이스가 실제로 어디로 갔는지]
- 예금 계좌 새로 열고 싶은데 → account
- 전기세 계좌 만들고 싶어 → account
- 관리비 내는 계좌 만들어줘 → account
- 수도요금 납부용 계좌 하나 필요해 → account
- 내 통장에 100만원 입금해줘 → account


In [ ]:
import os
import shutil
from datetime import datetime
from pathlib import Path
from tempfile import TemporaryDirectory
from uuid import uuid4

from langgraph.types import Command
from langgraph_finance_agent.graph import main_graph

SOURCE_DATA_DIR = Path.cwd() / "data"
USER_WITH_ACCOUNTS = "test-agent-01"   # data/*_test-agent-01.json 이 있는 유저
USER_NO_ACCOUNTS = "brand-new-99"      # 파일이 없는 유저 = 계좌 없음


def run_flow(question, answers=(), user_id=USER_WITH_ACCOUNTS, max_interrupts=4):
    """그래프를 끝까지 돌리고 어디로 가서 무슨 일이 있었는지 기록한다."""
    trace = []
    with TemporaryDirectory() as tmp:
        tmp_data = Path(tmp) / "data"
        tmp_data.mkdir()
        for f in SOURCE_DATA_DIR.glob(f"*_{user_id}.json"):
            shutil.copy(f, tmp_data / f.name)
        shutil.copy(SOURCE_DATA_DIR / "account_index.json", tmp_data / "account_index.json")

        cwd_before = os.getcwd()
        os.chdir(tmp)  # 데이터 경로가 상대경로("data")라 실행 위치를 임시 폴더로 옮긴다
        try:
            config = {"configurable": {"thread_id": str(uuid4())}}
            result = main_graph.invoke(
                {
                    "user_id": user_id,
                    "today": datetime.now().strftime("%Y-%m-%d"),
                    "request": question,
                },
                config,
            )

            answer_iter = iter(answers)
            while "__interrupt__" in result and len(trace) < max_interrupts:
                payload = result["__interrupt__"][0].value
                q_text = payload.get("question", "")
                trace.append(q_text.strip().splitlines()[0][:50] if q_text.strip() else "")

                ans = next(answer_iter, None)
                if ans is None:
                    break
                # CLI(__init__.py)와 동일: (y/n) 질문이면 bool, 아니면 문자열
                resume_value = (ans == "y") if "(y/n)" in q_text else ans
                result = main_graph.invoke(Command(resume=resume_value), config)
        finally:
            os.chdir(cwd_before)

    pending = "__interrupt__" in result
    return {
        "category": result.get("category"),
        "response": "" if pending else result.get("response", ""),
        "interrupts": trace,
        "pending": pending,                                # 끝까지 못 가고 interrupt에서 멈춤
        "loop": pending and len(trace) >= max_interrupts,  # 질문을 상한까지 받고도 안 끝남
    }


flow_cases = [
    {
        "name": "전기세 계좌 (정보 없음)",
        "question": "전기세 계좌 만들고 싶어",
        "answers": [],
        "user_id": USER_WITH_ACCOUNTS,
        "what_to_check": "어느 도메인? 안내문? unsupported?",
    },
    {
        "name": "전기세 계좌 + 고객번호, 새 계좌 요구 (계좌 있는 유저)",
        "question": "전기세 자동이체 등록할래, 고객번호 1234567890, 새 계좌 만들어서 연결하고 싶어",
        "answers": ["새 계좌 만들어줘"] * 3,
        "user_id": USER_WITH_ACCOUNTS,
        "what_to_check": "계좌 선택 interrupt에서 같은 질문이 반복되나? (loop=True면 탈출 불가)",
    },
    {
        "name": "전기세 자동이체 등록 (계좌 없는 유저)",
        "question": "전기세 자동이체 등록할래, 고객번호 1234567890",
        "answers": ["아무거나"] * 3,
        "user_id": USER_NO_ACCOUNTS,
        "what_to_check": "선택할 계좌가 없는데 어떻게 되나? ('연결할 계좌가 없어요'로 바로 끝나야 함)",
    },
    {
        "name": "입금 (미구현)",
        "question": "내 통장에 100만원 입금해줘",
        "answers": [],
        "user_id": USER_WITH_ACCOUNTS,
        "what_to_check": "account로 가는가? (입금이 이체 안내로 처리되는 것은 알려진 한계로 허용)",
    },
    {
        "name": "기준선: 단서 명확한 자동이체",
        "question": "가스비 자동이체 걸어줘",
        "answers": [],
        "user_id": USER_WITH_ACCOUNTS,
        "what_to_check": "정상적으로 bill_payment 안내문이 나오나?",
    },
    {
        "name": "계좌 선택 중 취소 (bill_payment)",
        "question": "전기세 자동이체 등록할래, 고객번호 1234567890",
        "answers": ["없는 계좌", "취소"],
        "user_id": USER_WITH_ACCOUNTS,
        "what_to_check": "'취소' 입력 시 루프를 빠져나와 '등록을 취소했어요'로 끝나는가?",
    },
]

for case in flow_cases:
    out = run_flow(case["question"], case["answers"], case["user_id"])
    print(f"\n=== {case['name']} ===")
    print(f"확인할 것: {case['what_to_check']}")
    print(f"category={out['category']}  pending={out['pending']}  loop={out['loop']}")
    print(f"interrupts={out['interrupts']}")
    print(f"response={out['response'][:120]!r}")


In [ ]:
from langsmith import Client, evaluate

client = Client()

ROUTER_DS = "finance-router-v1"
FLOW_DS = "finance-flow-v2"   # 케이스를 추가했으므로 v2 (같은 이름이면 업로드가 건너뛰어짐)


def upload_once(name, description, inputs, outputs, metadata):
    """같은 이름의 데이터셋이 이미 있으면 건드리지 않는다."""
    if client.has_dataset(dataset_name=name):
        print(f"{name}: 이미 있음 (내용을 바꿨다면 이름을 v2로 올리세요)")
        return
    ds = client.create_dataset(dataset_name=name, description=description)
    client.create_examples(dataset_id=ds.id, inputs=inputs, outputs=outputs, metadata=metadata)
    print(f"{name}: {len(inputs)}개 업로드")


# 라우터 데이터셋: 입력=발화, 정답=허용 도메인 목록, 메타데이터=태그
upload_once(
    ROUTER_DS,
    "최상위 라우터 도메인 분류 평가",
    inputs=[{"q": c["q"]} for c in router_set],
    outputs=[
        {"allowed": c["expected"] if isinstance(c["expected"], list) else [c["expected"]]}
        for c in router_set
    ],
    metadata=[{"tag": c["tag"]} for c in router_set],
)

# 흐름 데이터셋: 입력=발화+준비한 답+유저, 정답은 따로 없음(지표로 판단)
upload_once(
    FLOW_DS,
    "모호한 요청의 end-to-end 흐름 추적",
    inputs=[
        {"question": c["question"], "answers": c["answers"], "user_id": c["user_id"]}
        for c in flow_cases
    ],
    outputs=[{"note": c["what_to_check"]} for c in flow_cases],
    metadata=[{"name": c["name"]} for c in flow_cases],
)


In [11]:
# ── 라우터 평가 ──
def router_target(inputs: dict) -> dict:
    return {"routed": route(inputs["q"])}


def routed_correct(outputs: dict, reference_outputs: dict) -> dict:
    ok = outputs["routed"] in reference_outputs["allowed"]
    return {"key": "routed_correct", "score": float(ok)}


router_result = evaluate(
    router_target,
    data=ROUTER_DS,
    evaluators=[routed_correct],
    experiment_prefix="router",
    max_concurrency=8,
    metadata={"prompt_version": "baseline"},   # 프롬프트를 고칠 때마다 바꿔 두면 비교하기 쉬움
)


# ── 흐름 평가 ──
def flow_target(inputs: dict) -> dict:
    return run_flow(inputs["question"], inputs["answers"], inputs["user_id"])


def no_loop(outputs: dict) -> dict:
    return {"key": "no_loop", "score": float(not outputs["loop"])}


def completed(outputs: dict) -> dict:
    return {"key": "completed", "score": float(not outputs["pending"])}


flow_result = evaluate(
    flow_target,
    data=FLOW_DS,
    evaluators=[no_loop, completed],
    experiment_prefix="flow",
    max_concurrency=1,   # 중요: 아래 설명 참고
)

View the evaluation results for experiment: 'router-f7411094' at:
https://smith.langchain.com/o/5f26db71-ea58-4fb9-87bd-03acfd1a1bb3/datasets/03925206-fc34-41ec-8dea-1453c4f2e292/compare?selectedSessions=6683fdc3-e0df-4693-9dca-6742c41e13cc


View the evaluation results for experiment: 'flow-db3e0312' at:
https://smith.langchain.com/o/5f26db71-ea58-4fb9-87bd-03acfd1a1bb3/datasets/011c1107-b9b4-4736-9aba-62160c38db55/compare?selectedSessions=8ad9e18b-470a-4367-9c4d-ca77dfdb2a13


